In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
import pandas as pd
import loguru as logging

### Setting up dataset using custom dataset
Inheriting dataset class as a standard pracice in pytorch to create custom dataset.

In [ ]:
transform = transforms.Compose([transforms.CenterCrop(140,240),transforms.Resize(80,80,3),transforms.RandomCrop() ,transforms.ToTensor()])

In [ ]:
class GalaxyDataset(Dataset):
    ''' Custom dataset for galaxy classification weith weighted samplers. 
    Assume the mappping and label datasets are already merged'''

    CLASS_NAMES = {
    0: "completely_round_smooth",
    1: "in_between_smooth",
    2: "cigar_shaped_smooth",
    3: "edge_on",
    4: "spiral"}

    def __init__(self, img_dir,labels_df, transform=None,class_names=CLASS_NAMES):

        self.img_dir = img_dir
        self.labels =  labels_df
        self.transform = transform
        self.CLASS_NAMES = class_names

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        row = self.labels.iloc[idx]
        img_path = os.path.join(self.img_dir, row['objid'],".jpg")
        image = Image.open(img_path).convert('RGB')
        label = int(row['label'])  # Label column assumed to be named "label"

        if self.transform:
            image = self.transform(image)
        return image, label
    
    def make_weighted_sampler(self):

        class_counts = self.labels["label"].value_counts().sort_index()
        class_counts = torch.tensor(class_counts.values, dtype=torch.float)
        class_weights = 1.0 / class_counts
        sample_weights = class_weights[self.labels["label"]]

        return WeightedRandomSampler(
            weights     = sample_weights,
            num_samples = len(sample_weights),
            replacement = True
        )